In [7]:
import os

current_dir = os.getcwd()
print("Current working directory:", current_dir)

Current working directory: c:\Users\Admin\Desktop\Paraline\use_case\graph_rag


## Build Docker image 
Cài đặt môi trường vectorDB để phục vụ lưu trữ và truy vấn

In [12]:
!docker compose up -d 

 Container neo4j  Running


Cài đặt môi trường cần thiết

In [ ]:
!pip install llama-index neo4j openai python-dotenv gradio ipython llama-index-graph-stores-neo4j llama-index-llms-openai

In [14]:
!curl https://www.gutenberg.org/cache/epub/24022/pg24022.txt -o data_sample/book.txt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:--  0:00:01 --:--:--     0
  0     0    0     0    0     0      0      0 --:--:--  0:00:02 --:--:--     0Warning: Failed to open the file data_sample/book.txt: No such file or 

  0  184k    0     0    0     0      0      0 --:--:--  0:00:02 --:--:--     0
curl: (23) client returned ERROR on write of 14034 bytes


Đọc tài liệu và kiểm tra xem đã đọc được chưa

In [8]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader(input_files=["..//../data_sample/book.txt"]).load_data()

for doc in documents:
    print(doc.text[:300])

The Project Gutenberg eBook of A Christmas Carol
    
This ebook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg L


Kết nối Knowledge Graph DB, ở đây là NEO4j database, với ngôn ngữ truy vấn là Cypher

In [9]:
from llama_index.graph_stores.neo4j import Neo4jGraphStore
from llama_index.core import StorageContext, KnowledgeGraphIndex

# Kết nối Neo4j
graph_store = Neo4jGraphStore(
    username="neo4j",
    password="test1234",
    url="bolt://localhost:7687"
)

storage_context = StorageContext.from_defaults(graph_store=graph_store)




2025-09-10 15:48:15,380 - INFO - Received notification from DBMS server: {severity: INFORMATION} {code: Neo.ClientNotification.Schema.IndexOrConstraintAlreadyExists} {category: SCHEMA} {title: `CREATE CONSTRAINT IF NOT EXISTS FOR (e:Entity) REQUIRE (e.id) IS UNIQUE` has no effect.} {description: `CONSTRAINT constraint_1ed05907 FOR (e:Entity) REQUIRE (e.id) IS UNIQUE` already exists.} {position: None} for query: '\n                CREATE CONSTRAINT IF NOT EXISTS FOR (n:Entity) REQUIRE n.id IS UNIQUE;\n                '


Setup mô hình LLM và mô hình embedding, chúng ta sử dụng OpenAI

In [10]:
# %pip install llama-index llama-index-llms-openai
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
import os
from dotenv import load_dotenv
load_dotenv()
openai_api_key = os.environ.get("OPENAI_API_KEY")

llm = OpenAI(
    model="gpt-4o-mini",
    api_key=openai_api_key
)
embed_model = OpenAIEmbedding(
    api_key=openai_api_key
)


Embedding docs và tạo cơ sở dữ liệu trong knowledge Graph

In [25]:
# Tạo Knowledge Graph Index
kg_index = KnowledgeGraphIndex.from_documents(
    documents=documents,
    storage_context=storage_context,
    llm=llm,                  # chỉ để extract triplets
    embed_model=embed_model,  # để tạo embeddings
    include_embeddings=True,
    max_triplets_per_chunk=5,
    show_progress=True
)
kg_index.storage_context.persist(persist_dir="./graph_rag_storage")

Processing nodes:   0%|          | 0/59 [00:00<?, ?it/s]2025-09-10 12:22:53,302 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-09-10 12:22:54,630 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
Processing nodes:   2%|▏         | 1/59 [00:04<04:04,  4.22s/it]2025-09-10 12:22:57,052 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-09-10 12:22:58,249 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
Processing nodes:   3%|▎         | 2/59 [00:07<03:26,  3.62s/it]2025-09-10 12:23:00,450 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-09-10 12:23:01,169 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
Processing nodes:   5%|▌         | 3/59 [00:10<03:15,  3.49s/it]2025-09-10 12:23:03,436 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completio

##### Tạo query engine kết hợp KG và văn bản

In [ ]:
# Tạo query engine kết hợp KG và văn bản
query_engine = kg_index.as_query_engine(
    include_text=True,
    response_mode="tree_summarize",
    similarity_top_k=5,
    verbose=True,
)

In [12]:
# load lại KG index đã lưu
from llama_index.core import load_index_from_storage


graph_store = Neo4jGraphStore(
    username="neo4j",
    password="test1234",
    url="bolt://localhost:7687"
)

print("Loading Knowledge Graph Index from storage...")


# Load storage context từ folder đã save
storage_context = StorageContext.from_defaults(
    graph_store=graph_store,
    persist_dir="./graph_rag_storage"  # Same path bạn đã save
)

# Load index
kg_index = load_index_from_storage(storage_context)

print("✅ Successfully loaded Knowledge Graph Index!")

# Tạo query engine
query_engine = kg_index.as_query_engine(
    include_text=True,
    response_mode="tree_summarize",
    similarity_top_k=5,
    verbose=True
)

2025-09-10 15:48:55,998 - INFO - Received notification from DBMS server: {severity: INFORMATION} {code: Neo.ClientNotification.Schema.IndexOrConstraintAlreadyExists} {category: SCHEMA} {title: `CREATE CONSTRAINT IF NOT EXISTS FOR (e:Entity) REQUIRE (e.id) IS UNIQUE` has no effect.} {description: `CONSTRAINT constraint_1ed05907 FOR (e:Entity) REQUIRE (e.id) IS UNIQUE` already exists.} {position: None} for query: '\n                CREATE CONSTRAINT IF NOT EXISTS FOR (n:Entity) REQUIRE n.id IS UNIQUE;\n                '
2025-09-10 15:48:56,045 - INFO - Loading all indices.


Loading Knowledge Graph Index from storage...
Loading llama_index.core.storage.kvstore.simple_kvstore from ./graph_rag_storage\docstore.json.
Loading llama_index.core.storage.kvstore.simple_kvstore from ./graph_rag_storage\index_store.json.
✅ Successfully loaded Knowledge Graph Index!


Thực hiện truy vấn

In [13]:
# Thực hiện truy vấn
response = query_engine.query("Who is Fred")
print(response)

2025-09-10 15:49:09,177 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Extracted keywords: ['Fred']


2025-09-10 15:49:10,398 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-09-10 15:49:10,437 - INFO - > Querying with idx: c423dfb9-4d67-4c19-92e8-9fc8012a291d: At every
fresh question that was put to him, this nephew burst into a fresh ...
2025-09-10 15:49:10,439 - INFO - > Querying with idx: a46e95fd-c670-443a-b195-e9c6a98cc8de: Master Scrooge's trunk being by this time tied on to the top of the
chaise, ...
2025-09-10 15:49:10,440 - INFO - > Querying with idx: cca738d1-6cd5-4b20-8d90-52b80bf249af: 'Thankee,' said Scrooge. 'I am much obliged to you. I thank you fifty
times....
2025-09-10 15:49:10,441 - INFO - > Querying with idx: 6d4a881e-0a77-456e-8c08-265308a174d1: Fezziwig. As to _her_, she
was worthy to be his partner in every sense of th...
2025-09-10 15:49:10,442 - INFO - > Querying with idx: 15bc62fc-35dc-48c7-9c9f-7ee800e9c418: Hilli-ho, Dick! Chirrup, Ebenezer!'

Clear away! There was nothing they wou...
2025-09-10 15:49:10,443 - INFO - >

KG context:
The following are knowledge sequence in max depth 2 in the form of directed graph like:
`subject -[predicate]->, object, <-[predicate_next_hop]-, object_next_hop ...`
['KNOWS', 'Uncle scrooge', 'HAS_GIVEN', 'Plenty of merriment']
('Bob', 'Said', 'He is pleasantest-spoken gentleman')
('Fezziwig', 'Is', 'Old gentleman')
('Fred', 'Knows', 'Uncle scrooge')
('Scrooge', 'Is', 'Uncle of fred')
['KNOWS', 'Uncle scrooge']
('Fezziwig', 'Is', 'Old')


2025-09-10 15:49:11,421 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Fred is the nephew of Uncle Scrooge.


UI cho việc test

In [ ]:
import gradio as gr
from llama_index.core.memory import ChatMemoryBuffer
from llama_index.core.llms import ChatMessage

memory = ChatMemoryBuffer.from_defaults(token_limit=2000)

def conversational_query(message: str):
    # Lấy lịch sử hội thoại từ memory
    past_messages = memory.get()
    context_text = "\n".join([f"{m.role}: {m.content}" for m in past_messages])

    # Xây prompt giống LangChain
    prompt = (
        f"You are a helpful assistant. "
        f"Here is the conversation so far:\n{context_text}\n\n"
        f"User: {message}\nAssistant:"
    )

    # Query từ KG
    response = query_engine.query(prompt)
    response_str = str(response)

    # Lưu vào memory
    memory.put(ChatMessage(role="user", content=message))
    memory.put(ChatMessage(role="assistant", content=response_str))

    return response_str

# ===============================
# Giao diện Gradio
# ===============================
def chat_fn(message, history):
    response = conversational_query(message)

    # cập nhật UI history (format messages)
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": response})
    return "", history

with gr.Blocks() as demo:
    gr.Markdown("## 📚 Conversational KG Chat (LlamaIndex style)")

    chatbot = gr.Chatbot(type="messages", height=500)
    msg = gr.Textbox(label="Your message")

    msg.submit(chat_fn, [msg, chatbot], [msg, chatbot])

    # Nút Clear
    clear_btn = gr.Button("Clear Chat")

    def clear_fn():
        memory.reset()
        return []

    clear_btn.click(clear_fn, outputs=[chatbot])

demo.launch(share=True)


c:\Users\Admin\anaconda3\envs\RAG\Lib\site-packages\gradio\http_server.py:120: ResourceWarning: unclosed <socket.socket fd=5784, family=2, type=1, proto=0>
  s = socket.socket()
c:\Users\Admin\anaconda3\envs\RAG\Lib\site-packages\gradio\http_server.py:120: ResourceWarning: unclosed <socket.socket fd=2396, family=2, type=1, proto=0>
  s = socket.socket()
2025-09-08 11:17:59,119 - INFO - HTTP Request: GET http://127.0.0.1:7866/gradio_api/startup-events "HTTP/1.1 200 OK"
2025-09-08 11:17:59,151 - INFO - HTTP Request: HEAD http://127.0.0.1:7866/ "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7866


2025-09-08 11:17:59,897 - INFO - HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
2025-09-08 11:17:59,984 - INFO - HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"



Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
